In [123]:
# ALIGNED DURATIONS
import torch


def extract_durations_from_alignment(text_features, mel_spectrogram, text_lengths):
    """
    Extract ground truth durations by computing attention between text and mel.

    Args:
        text_features: [B, hidden, T_text] - encoded text embeddings
        mel_spectrogram: [B, n_mels, T_mel] - target mel spectrogram
        text_lengths: [B] - actual (unpadded) text lengths

    Returns:
        durations: [B, T_text] - ground truth durations (how many mel frames per token)
    """
    B, C, T_text = text_features.shape
    _, n_mels, T_mel = mel_spectrogram.shape

    text_norm = F.normalize(text_features, dim=1)

    mel_to_hidden = nn.Linear(n_mels, C).to(text_features.device)
    mel_proj = mel_to_hidden(mel_spectrogram.transpose(1, 2))
    mel_proj = F.normalize(mel_proj, dim=2)

    # Compute similarity matrix: [B, T_text, T_mel]
    similarity = torch.bmm(text_norm.transpose(1, 2), mel_proj.transpose(1, 2))

    # Apply softmax over mel dimension (each text token attends to mel frames)
    attention = F.softmax(similarity, dim=2)

    # Extract durations: for each text token, count how many mel frames it "owns"
    # Use argmax to find the best matching mel frame for each text token
    # Then count consecutive assignments
    durations = torch.zeros(B, T_text, device=text_features.device, dtype=torch.long)

    for b in range(B):
        text_len = text_lengths[b].item()

        # Get attention weights for this batch item
        attn = attention[b, :text_len, :]

        # For each mel frame, find which text token it attends to most
        mel_to_text = torch.argmax(attn, dim=0)

        # Count how many mel frames are assigned to each text token
        for t in range(text_len):
            durations[b, t] = (mel_to_text == t).sum()

    durations = durations.clamp(min=1)

    for b in range(B):
        text_len = text_lengths[b].item()
        total_dur = durations[b, :text_len].sum()
        diff = T_mel - total_dur

        if diff > 0:
            durations[b, text_len - 1] += diff
        elif diff < 0:
            durations[b, text_len - 1] = max(1, durations[b, text_len - 1] + diff)

    return durations

def get_aligned_durations(text_len, mel_len, batch_size, device):
    """
    Creates durations that sum EXACTLY to mel_len.
    This guarantees the upsampled tensor perfectly matches the target mel length.
    """
    base_dur = mel_len // text_len
    remainder = mel_len % text_len

    # Start with base duration for all tokens
    durs = torch.full((batch_size, text_len), base_dur, device=device, dtype=torch.long)

    # Distribute the remainder to the first few tokens to make the sum exact
    for b in range(batch_size):
        durs[b, :remainder] += 1

    return durs

In [129]:
import torchaudio
# TRAINING
from tqdm import tqdm
from transformers import VitsTokenizer

tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-rus")
vocab_size = len(tokenizer)

num_epochs = 1000
save_every = 1000
global_step = 0

min_audio_length = 1024
mel_hop_length = 512

device = "cuda:0" if torch.cuda.is_available() else "cpu"

mel_extractor = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    # number of digital samples analyzed in a single window of time
    n_fft=1024,
    win_length=min_audio_length,
    hop_length=mel_hop_length,
    n_mels=80,
    f_min=0,
    f_max=8000,
).to(device)

model = SimpleVITS(vocab_size).to(device)

# Separate the optimizers as your code requires
optimizer_g = torch.optim.AdamW(model.generator.parameters(), lr=5e-5, betas=(0.8, 0.99))
optimizer_d = torch.optim.AdamW(model.discriminator.parameters(), lr=5e-5, betas=(0.8, 0.99))

model.train()

with open("sonya_dataset/index.json", 'r', encoding='utf-8') as f:
    index = json.load(f)
    data = [(audio_path, index[audio_path]) for audio_path in index]

dataset = VitsAudioDataset(data, tokenizer, mel_extractor)
data_loader = create_data_loader(dataset)

if not os.path.exists("./checkpoints"):
    os.mkdir("./checkpoints")

for epoch in range(num_epochs):
    progres_bar = tqdm(data_loader, desc="Epoch {}".format(epoch + 1))
    for batch in progres_bar:
        losses = train_step(batch, model, optimizer_g, optimizer_d, device, mel_extractor)

        progres_bar.set_postfix({
            "G_loss": f"{losses['g_loss']:.4f}",
            "D_loss": f"{losses['d_loss']:.4f}",
            "Mel": f"{losses['mel_loss']:.4f}",
            "dur_loss": f"{losses['dur_loss']:.4f}",
        })

        global_step += 1

        if global_step % save_every == 0:
            checkpoint = model.generator.state_dict()

            torch.save(checkpoint, f"./checkpoints/checkpoint-{global_step}.pth")
            print(f"\n✅ Saved checkpoint at step {global_step}")

Epoch 1:   0%|          | 0/65 [00:23<?, ?it/s]


ValueError: too many values to unpack (expected 2)

In [148]:
import numpy as np
from scipy.io import wavfile
from src.model.generator import SimplifiedVITSGenerator

# RUNNING
model = SimplifiedVITSGenerator(vocab_size=vocab_size).to(device)

point = 2

model.load_state_dict(torch.load(f"./checkpoints/checkpoint-{point}000.pth", map_location=device))
model.eval()  # CRITICAL: Sets dropout/batchnorm to inference mode


def generate_speech(text, output_path="output.wav"):
    print(f"Generating speech for: '{text}'")

    # 1. Tokenize
    encoding = tokenizer(text, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)

    # 2. Generate (training=False triggers duration prediction)
    with torch.no_grad():
        output = model(input_ids, training=False)
    audio_tensor = output['audio_generated']
    log_durs = output['audio_generated']

    # Debug: Print raw log durations and converted durations
    print(f"Raw log durations: {log_durs.squeeze().cpu().numpy()}")
    print(f"Min: {log_durs.min().item():.2f}, Max: {log_durs.max().item():.2f}, Mean: {log_durs.mean().item():.2f}")

    raw_durs = torch.exp(log_durs)
    print(f"Exp(log_durs): {raw_durs.squeeze().cpu().numpy()}")

    durs = torch.round(raw_durs).clamp(min=1, max=30).long()
    print(f"Final durations: {durs.squeeze().cpu().numpy()}")
    print(f"Total mel frames: {durs.sum().item()}")

    # 3. Post-process audio
    # audio_tensor is [1, 1, Time]. Squeeze to [Time]
    audio_np = audio_tensor.squeeze().cpu().numpy()

    # Ensure it's in [-1, 1] range, then scale to 16-bit PCM
    audio_np = np.clip(audio_np, -1.0, 1.0)
    audio_int16 = (audio_np * 32767).astype(np.int16)

    # 4. Save to WAV
    sample_rate = 16000
    wavfile.write(output_path, sample_rate, audio_int16)
    print(f"✅ Successfully saved to: {output_path}")


test_text = "Привет! Это тест синтезированной речи на чистой архитектуре PyTorch."
generate_speech(test_text, f"my_first_tts_output_{point}.wav")

Generating speech for: 'Привет! Это тест синтезированной речи на чистой архитектуре PyTorch.'


OutOfMemoryError: CUDA out of memory. Tried to allocate 91.76 GiB. GPU 0 has a total capacity of 11.94 GiB of which 9.21 GiB is free. Of the allocated memory 736.51 MiB is allocated by PyTorch, and 803.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [145]:
import torch

a = torch.zeros( 2,3,4)
b= a[0, :2, :]
# b
torch.argmax(b, dim=0)

tensor([0, 0, 0, 0])